In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

In [2]:
import torch
import torch.nn as nn

# torch.autograd.set_detect_anomaly(True)

In [12]:

import torch
import torch.nn as nn

# torch.autograd.set_detect_anomaly(True)

class TRL(nn.Module):
    def __init__(self, input_size, output, rank, ignore_modes = (0,), bias = True, device = 'cuda'):
        super(TRL, self).__init__()
        
        alphabet = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQERSUVWXYZ'
        self.device = device
        self.bias = bias
        
        if isinstance(input_size, int):
            self.input_size = (input_size, )
        else:
            self.input_size = tuple(input_size)
            
        if isinstance(output, int):
            self.output = (output, )
        else:
            self.output = tuple(output)
        
        if isinstance(rank, int):
            self.rank = (rank, )
        else:
            self.rank = tuple(rank)

        if isinstance(ignore_modes, int):
            self.ignore_modes = (rank, )
        else:
            self.ignore_modes = tuple(ignore_modes)
        
        
        # remove ignored modes from the input size
        new_size = []
        for i in range(len(self.input_size)):
            if i in self.ignore_modes:
                continue
            else:
                new_size.append(self.input_size[i])
        
        self.w_size = tuple(new_size) + self.output
        if self.bias:
            self.register_parameter('b', nn.Parameter(torch.randn(self.output, device = self.device), requires_grad=True))
        else:
            self.register_parameter('b',None)
            
        # Tucker Decomposition method for TRL
        
        self.register_parameter('core', nn.Parameter(torch.randn(self.rank, device = self.device), requires_grad=True))

        # List of all factors
        for i,r in enumerate(self.rank):
            self.register_parameter(f'u{i}', nn.Parameter(torch.randn((r, self.w_size[i]), device = self.device), requires_grad=True))      

        # Generate formula for w :
        
        index = 0
        formula = ''
        core_str = ''
        w_str = ''
        for i in range(len(self.core.shape)):
            formula+=alphabet[index]
            index+=1
            if i== len(self.core.shape) - 1:
                formula+=','
        core_str = formula[:len(formula)-1]
                
        for l in range(len(self.rank)):
            formula+=core_str[l]
            formula+=alphabet[index]
            w_str+=alphabet[index]
            index+=1
            if l < len(self.rank) - 1:
                formula+=','
            elif l == len(self.rank) - 1:
                    formula+='->'
        
        formula+=w_str
        print(formula)
        
        self.w_formula = formula   
        operands = [self.core]
        for i in range(len(self.rank)):
            operands.append(getattr(self, f'u{i}'))  

        self.w_operands = operands
        # self.w = torch.einsum(self.w_formula, operands)
        
        # Generate formula for Generalized Inner Product of W and X:
        index = 0
        formula = ''
        mul = ''
        out_str = ''
        extend_str =''
        for i in range(len(self.input_size)):
            formula+=alphabet[index]
            if i not in self.ignore_modes:
                mul+= alphabet[index]
            else:
                extend_str+= alphabet[index]
            index+=1
            if i== len(self.input_size) - 1:
                formula+=','
        
        formula+=mul
        for i in range(len(mul),len(self.w_size)):
            formula+=alphabet[index]
            out_str+=alphabet[index]
            index+=1
            if i== len(self.w_size) - 1:
                formula+='->'
         
        formula+=extend_str+out_str       
        self.out_formula = formula
        print(formula)
        
    def forward(self, x):
        w = torch.einsum(self.w_formula, self.w_operands)
        out = torch.einsum(self.out_formula, (x, w))
        if self.bias:
            out += self.b 
        return out # You may rearrange your out tensor to your desired shapes

In [13]:

import torch
import torch.nn as nn

class TCL(nn.Module):
    def __init__(self, input_size, rank, ignore_modes = (0,), bias = True, device = 'cuda'):
        super(TCL, self).__init__()
        
        alphabet = 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQERSUVWXYZ'
        self.device = device
        self.bias = bias
        
        if isinstance(input_size, int):
            self.input_size = (input_size, )
        else:
            self.input_size = tuple(input_size)
        
        if isinstance(rank, int):
            self.rank = (rank, )
        else:
            self.rank = tuple(rank)
        
        if isinstance(ignore_modes, int):
            self.ignore_modes = (rank, )
        else:
            self.ignore_modes = tuple(ignore_modes)
        
        # remove ignored modes from the input size
        new_size = []
        for i in range(len(self.input_size)):
            if i in self.ignore_modes:
                continue
            else:
                new_size.append(self.input_size[i])
        
        if self.bias:
            self.register_parameter('b', nn.Parameter(torch.randn(self.rank, device=self.device), requires_grad=True))
            self.b = nn.Parameter(torch.randn(self.rank), requires_grad=True)
        else:
            self.register_parameter('b',None)
            
        # Tucker Decomposition method for TCL
                                   
        # List of all factors
        for i,r in enumerate(self.rank):
            self.register_parameter(f'u{i}', nn.Parameter(torch.randn((r, new_size[i]), device = self.device), requires_grad=True))

        # Generate formula for output :
        index = 0
        formula = ''
        core_str = ''
        extend_str = ''
        out_str = ''
        for i in range(len(self.input_size)):
            formula+=alphabet[index]
            if i not in self.ignore_modes:
                core_str+=alphabet[index]
            else:
                extend_str+=alphabet[index]   
            index+=1
            if i==len(self.input_size)-1:
                formula+=','
        
        for l in range(len(self.rank)):
            formula+=core_str[l]
            formula+=alphabet[index]
            out_str+=alphabet[index]
            index+=1
            if l < len(self.rank) - 1:
                formula+=','
            elif l == len(self.rank) - 1:
                    formula+='->'
        formula+=extend_str+out_str  
            
        self.out_formula = formula
        # print(formula)        
        
    def forward(self, x):
        operands = [x]
        for i in range(len(self.rank)):
            operands.append(getattr(self, f'u{i}'))  

        self.w_operands = operands
        out = torch.einsum(self.out_formula, operands)
        if self.bias:
            out += self.b
        return out # You may rearrange your out tensor to your desired shapes 
    


In [7]:
# Check Test Plan for more details 
# Test ResNet50 model on MNIST dataset
# New Classifier - TCL/TRL Model Our Method
# Optimizer Adam - Default
# No Scheduler
# MNIST dataset -> (3, 192, 192) 
# Pretrained
# Trasfer Learning
# Without Adaptive avg pooling
########################################################

# Add all .py files to path
import sys
sys.path.append('..')

# Import Libraries
# from Utils.Accuracy_measures import topk_accuracy
# from Utils.Mnist_loader import get_mnist_dataloaders
# from Utils.Num_parameter import count_parameters
# from Models.Resnet50 import Resnet50
from TRL import TRL
from TCL import TCL
import tltorch


import torchvision.transforms as transforms
from torch import nn
from torch import optim

import time
import torch
import os

In [5]:
# Setup the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
print(f'Device is set to : {device}')

# Set up the transforms and train/test loaders
# image_size = 192

# mnist_transform_train = transforms.Compose([
#         transforms.Grayscale(num_output_channels=3),
#         transforms.RandomHorizontalFlip(),
#         transforms.RandomCrop(32, padding=2),
#         transforms.Resize((image_size, image_size)), 
#         transforms.ToTensor(),
#         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
#     ])

# mnist_transform_test = transforms.Compose([
#         transforms.Resize((image_size, image_size)), 
#         transforms.Grayscale(num_output_channels=3), 
#         transforms.ToTensor(),
#         transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
#     ])


# train_loader, test_loader = get_mnist_dataloaders(
#                                     data_dir = '../datasets',
#                                     batch_size = 2,
#                                     image_size = 192,
#                                     transform_train = mnist_transform_train ,
#                                     transform_test = mnist_transform_test)

Device is set to : cuda


In [8]:
x1 = torch.rand((16,10)).to(device)
x2 = torch.rand((16,10)).to(device)
x3 = torch.rand((16,5)).to(device)


new_classifier = nn.Sequential(
    TCL(input_size=(16,10), rank=(10), ignore_modes=(0,), bias=True, device= device),
    TRL(input_size=(16,10), output=(5), rank=(10,5),ignore_modes = (0,), bias = True, device = device),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(new_classifier.parameters())


optimizer.zero_grad()    
outputs = new_classifier(x1)
loss = criterion(x3, outputs)
for p in new_classifier.parameters():
    print(p.shape)
# loss.requires_grad = True
loss.backward(retain_graph=True)
optimizer.step()

optimizer.zero_grad()    
outputs = new_classifier(x2)
loss = criterion(x3, outputs)
for p in new_classifier.parameters():
    print(p.shape)
# loss.requires_grad = True
loss.backward(retain_graph=True)
optimizer.step()


torch.Size([10])
torch.Size([10, 10])
torch.Size([5])
torch.Size([10, 5])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([10])
torch.Size([10, 10])
torch.Size([5])
torch.Size([10, 5])
torch.Size([10, 10])
torch.Size([5, 5])


In [9]:
x1 = torch.rand((2,2048,6,6)).to(device)
x2 = torch.rand((2,2048,6,6)).to(device)
x3 = torch.rand((2,10)).to(device)


new_classifier = nn.Sequential(
    TCL(input_size=(2,2048,6,6), rank=(2048,6,6), ignore_modes=(0,), bias=True, device=device),
    TRL(input_size=(2,2048,6,6), output=(10), rank=(200,3,3,8),ignore_modes = (0,), bias = True, device = device),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(new_classifier.parameters())


optimizer.zero_grad()    
outputs = new_classifier(x1)
loss = criterion(x3, outputs)
for p in new_classifier.parameters():
    print(p.shape)
# loss.requires_grad = True
loss.backward()
optimizer.step()

optimizer.zero_grad()    
outputs = new_classifier(x2)
loss = criterion(x3, outputs)
for p in new_classifier.parameters():
    print(p.shape)
# loss.requires_grad = True
loss.backward()
optimizer.step()



torch.Size([2048, 6, 6])
torch.Size([2048, 2048])
torch.Size([6, 6])
torch.Size([6, 6])
torch.Size([10])
torch.Size([200, 3, 3, 8])
torch.Size([200, 2048])
torch.Size([3, 6])
torch.Size([3, 6])
torch.Size([8, 10])
torch.Size([2048, 6, 6])
torch.Size([2048, 2048])
torch.Size([6, 6])
torch.Size([6, 6])
torch.Size([10])
torch.Size([200, 3, 3, 8])
torch.Size([200, 2048])
torch.Size([3, 6])
torch.Size([3, 6])
torch.Size([8, 10])


In [5]:
# Set up the new classifier 

# from TRL import TRL


new_classifier = nn.Sequential(
    TRL(input_size=(2,2048,6,6), output=(10), rank=(200,3,3,8),ignore_modes = (0,), bias = False, device = device),
)

# new_classifier = nn.Sequential(
#     tltorch.TRL(input_shape=(2048,6,6), output_shape=(10), factorization='Tucker', rank=(200,3,3,8))
# )
    

# Set up the model, optimizer and criterion
model = Resnet50(pretrained=True,
                        weights_path='../weights/resnet50_weights.pth',
                        tensorized=True,
                        input_shape=(192,192),
                        num_classes=10,
                        avg_pool=False,
                        new_classifier=new_classifier).to(device)

# Load pretrained from Tests


num_parameters = count_parameters(model)
classifier_parameters = count_parameters(model.classifier)
print(f'This Model has {num_parameters} parameters')
print(f'This Model has {classifier_parameters} classifier parameters')



criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

for _, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()    
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        for p in model.parameters():
            if p.grad is not None:
                print('NONE NIST')

        loss.backward(retain_graph=True)
        for p in model.parameters():
            if p.grad is not None:
                p.grad=p.grad.to(device)
        print(_)
        optimizer.step()      

abcd,ae,bf,cg,dh->efgh
abcd,bcde->ae
This Model has 23932148 parameters
This Model has 424116 classifier parameters
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
24

KeyboardInterrupt: 

In [10]:
# Define train and test functions (use examples)
def train_epoch(loader, epoch):
    model.train()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
    
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        loss.backward()

        for p in model.classifier.parameters():
            if p.grad is not None:
                print(p.grad.device)
                # p.grad=p.grad.to(device)
        optimizer.step()
        

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_train)

    return report_train

def test_epoch(loader, epoch):
    model.eval()

    start_time = time.time()
    running_loss = 0.0
    correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

    for _, (inputs, targets) in enumerate(loader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, targets)

        running_loss += loss.item()
        accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
        for k in accuracies:
            correct[k] += accuracies[k]['correct']

    elapsed_time = time.time() - start_time
    top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
    avg_loss = running_loss / len(loader.dataset)

    report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
    print(report_test)

    return report_test

In [11]:
# Set up the directories to save the results
TEST_ID = 'Test_ID091-deleteme'
result_dir = os.path.join('../results', TEST_ID)
result_subdir = os.path.join(result_dir, 'accuracy_stats')
model_subdir = os.path.join(result_dir, 'model_stats')

os.makedirs(result_subdir, exist_ok=True)
os.makedirs(model_subdir, exist_ok=True)

with open(os.path.join(result_dir, 'model_stats', 'model_info.txt'), 'a') as f:
    f.write(f'total number of parameters:\n{num_parameters}\ntotal number of classifier parameters:\n{classifier_parameters}')

# Freeze Convolutional Layers
layer = 0
for child in model.children():
    layer+=1
    if layer < 3:
        for param in child.parameters():
            param.requires_grad = False

# Train and Test The Model - Frozen Layers
n_epoch = 1
print(f'Training for {len(range(n_epoch))} epochs\n')
for epoch in range(1,n_epoch+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)

    report = report_train + '\n' + report_test + '\n\n'
    if epoch % 10 == 0:
        model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        torch.save(model.state_dict(), model_path)
    with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
        f.write(report)
        
# Unfreeze all layers
layer = 0
for child in model.children():
    layer+=1
    if layer < 3:
        for param in child.parameters():
            param.requires_grad = True
            
# Train and Test The Model - Unfrozen Layers - comment if not required
n_epoch_additional = 0
print(f'Training for Additional {len(range(n_epoch_additional))} epochs\n')
for epoch in range(n_epoch+1,n_epoch+n_epoch_additional+1):
    report_train = train_epoch(train_loader, epoch)
    report_test = test_epoch(test_loader, epoch)

    report = report_train + '\n' + report_test + '\n\n'
    if epoch % 5 == 0:
        model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        torch.save(model.state_dict(), model_path)
    with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
        f.write(report)


Training for 30 epochs

cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cuda:0
cud

KeyboardInterrupt: 